In [1]:
import os, sys
sys.path.append("..")

from src.milvus_store import MilvusStore
from src.config import ConfigLoader
from src.agent import AgenticRAG

2025-10-18 14:16:46,638 - src.agent - INFO - Logging to file: logs/agent.log


In [2]:
config = ConfigLoader("../config.yaml")

In [3]:
config.set("database", "collection_name", "doc_2_db")

uri = config.get("database", "uri", default="http://localhost:19530")
db_name = config.get("database", "name", default="gil")
collection_name = config.get("database", "collection_name", default="multimodal_rag")
embed_model = config.get("model", "embeddings", default="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
# namespace = config.get("database", "namespace", default="viettel")


vs1 = {"store": MilvusStore(
            uri=uri,
            db_name=db_name,
            collection_name=collection_name,
            drop_old=False,
          #   namespace=namespace
            ),
       "name": "retrieve_documents_on_docling",
       "description": "Search and retrieve information from the document collection on a topic of docling",
       "k": config.get("retrieval", "k", default=3),
       "ranker_weights": config.get("retrieval", "weights", default=[0.6, 0.4])
       }

vector_stores = [vs1]

/Users/github/rag-multimodal/notebooks/../src/milvus_store.py:39: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings_model = HuggingFaceEmbeddings(model_name=self.embed_model)


In [4]:
# milvus_store.similarity_search_with_score(question)

In [5]:
agent = AgenticRAG(vector_stores=vector_stores)

2025-10-18 14:16:54,982 - src.agent - INFO - Graph visualization saved to optimized_graph.png
2025-10-18 14:16:54,984 - src.agent - INFO - OptimizedAgenticRAG initialized with model: qwen3:1.7b and 1 vector store(s)


In [6]:
results = agent.run_mcq_csv("/Users/github/rag-multimodal/test.csv")

2025-10-18 14:16:54,994 - src.agent - INFO - Processing CSV row 1: Năm 2010, công nghệ nào thường được sử dụng trong ...
2025-10-18 14:16:54,995 - src.agent - INFO - Thread ID updated to: 1760771814994_a4a48386792c409f8e402e96ccdc39c6
2025-10-18 14:16:54,995 - src.agent - INFO - Running MCQ with HNSW optimization: Năm 2010, công nghệ nào thường được sử dụng trong kiểm soát ra vào ngôi nhà thông minh?...
2025-10-18 14:16:54,996 - src.agent - INFO - MCQ options: ['A', 'B', 'C', 'D']
2025-10-18 14:16:54,996 - src.agent - INFO - Thread ID updated to: 1760771814996_a1dc40068b964a92a254a210899afcd6
2025-10-18 14:16:55,000 - src.agent - INFO - [trace] _generate_query_or_respond called
2025-10-18 14:16:58,407 - src.agent - INFO - Reset retriever filters
2025-10-18 14:16:58,414 - src.agent - INFO - MCQ options detected: ['A', 'B', 'C', 'D']
2025-10-18 14:17:02,524 - src.agent - INFO - Retrieved docs count: 0
2025-10-18 14:17:02,526 - src.agent - INFO - Context length: 237
2025-10-18 14:17:02,86

In [7]:
from pprint import pprint
import re

In [8]:
results

[{'question': 'Năm 2010, công nghệ nào thường được sử dụng trong kiểm soát ra vào ngôi nhà thông minh?',
  'A': 'Bluetooth Low Energy',
  'B': 'RFID và nhận dạng khuôn mặt/vân tay',
  'C': 'Zigbee',
  'D': 'Blockchain',
  'response': 'Thinking: Câu hỏi hỏi về công nghệ nào được sử dụng trong kiểm soát ra vào ngôi nhà thông minh năm 2010. Các lựa chọn bao gồm Bluetooth Low Energy, RFID và nhận dạng khuôn mặt/vân tay, Zigbee, Blockchain. Năm 2010, Zigbee là công nghệ phổ biến trong các hệ thống thông minh, đặc biệt là các hệ thống nhà thông minh. Bluetooth Low Energy còn được phát triển vào những năm gần đây, còn RFID và nhận dạng khuôn mặt/vân tay là công nghệ đã được sử dụng từ lâu. Do đó, đáp án đúng là C: Zigbee.\nRationale: Zigbee là công nghệ phổ biến trong các hệ thống thông minh, đặc biệt là các hệ thống nhà thông minh. Bluetooth Low Energy còn được phát triển vào những năm gần đây, còn RFID và nhận dạng khuôn mặt/vân tay là công nghệ đã được sử dụng từ lâu.\nFinal Answer: C'},
 

In [10]:
for r in results:
    response = r["response"]

    # --- Tách phần <think> ... </think>
    thinking = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    thinking_text = thinking.group(1).strip() if thinking else ""

    # --- Phần còn lại sau </think>
    remainder = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()

    # --- Tách rationale và final answer (nếu có)
    rationale_match = re.search(r"Rationale:(.*?)(?:Final Answer:|$)", remainder, re.DOTALL)
    rationale_text = rationale_match.group(1).strip() if rationale_match else ""

    final_match = re.search(r"Final Answer:\s*(.*)", remainder)
    final_answer = final_match.group(1).strip() if final_match else ""

    # --- In ra kết quả
    print(f"\n❓ Câu hỏi: {r['question']}\n")
    print("=== 🧠 Thinking ===")
    pprint(thinking_text)

    print("\n=== 💬 Final Answer ===")
    print(final_answer)

    print("\n=== 📚 Rationale ===")
    pprint(rationale_text)
    print("-" * 80)



❓ Câu hỏi: Năm 2010, công nghệ nào thường được sử dụng trong kiểm soát ra vào ngôi nhà thông minh?

=== 🧠 Thinking ===
''

=== 💬 Final Answer ===
C

=== 📚 Rationale ===
('Zigbee là công nghệ phổ biến trong các hệ thống thông minh, đặc biệt là các '
 'hệ thống nhà thông minh. Bluetooth Low Energy còn được phát triển vào những '
 'năm gần đây, còn RFID và nhận dạng khuôn mặt/vân tay là công nghệ đã được sử '
 'dụng từ lâu.')
--------------------------------------------------------------------------------

❓ Câu hỏi: Nội dung chính của bài báo nghiên cứu về IoT trong xây dựng nhà thông minh là gì?

=== 🧠 Thinking ===
''

=== 💬 Final Answer ===
C

=== 📚 Rationale ===
('Bài báo nghiên cứu về IoT trong xây dựng nhà thông minh tập trung vào việc '
 'phân loại và đánh giá các nghiên cứu IoT ứng dụng trong lĩnh vực nhà thông '
 'minh trong 10 năm qua. Câu hỏi yêu cầu nội dung chính của bài báo, nên lựa '
 'chọn liên quan đến việc phân loại và đánh giá các nghiên cứu. Lựa chọn C phù '
 'hợp nhấ

In [9]:
for r in results:
    response = r["response"]

    # --- Tách phần <think> ... </think>
    thinking = re.search(r"<think>(.*?)</think>", response, re.DOTALL)
    thinking_text = thinking.group(1).strip() if thinking else ""

    # --- Phần còn lại sau </think>
    remainder = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()

    # --- Tách rationale và final answer (nếu có)
    rationale_match = re.search(r"Rationale:(.*?)(?:Final Answer:|$)", remainder, re.DOTALL)
    rationale_text = rationale_match.group(1).strip() if rationale_match else ""

    final_match = re.search(r"Final Answer:\s*(.*)", remainder)
    final_answer = final_match.group(1).strip() if final_match else ""

    # --- In ra kết quả
    print(f"❓ Câu hỏi: {r['question']}")
    print("=== 💬 Final Answer ===")
    print(final_answer)
    print("-" * 80)

❓ Câu hỏi: Trong mô hình nhà thông minh, IoT chủ yếu đóng vai trò gì?
=== 💬 Final Answer ===
B
--------------------------------------------------------------------------------
❓ Câu hỏi: Năm 2010, công nghệ nào thường được sử dụng trong kiểm soát ra vào ngôi nhà thông minh?
=== 💬 Final Answer ===
C
--------------------------------------------------------------------------------
❓ Câu hỏi: Bộ cảm biến trong nhà thông minh thực hiện chức năng gì và bộ truyền động có vai trò gì?
=== 💬 Final Answer ===

--------------------------------------------------------------------------------
